In [ ]:
import os
import shutil

# --- Paths ---
SOURCE_DIR = "/kaggle/input/food-101/food-101"
SRC_TRAIN = os.path.join(SOURCE_DIR, "train")
SRC_VAL = os.path.join(SOURCE_DIR, "validation")
DEST_DIR = "/kaggle/working/food-30"  # must be in /kaggle/working to be writable

# --- Pick 30 classes alphabetically ---
all_labels = sorted([d for d in os.listdir(SRC_TRAIN) if os.path.isdir(os.path.join(SRC_TRAIN, d))])
selected_labels = all_labels[:30]  # first 30 alphabetically

print(f"📁 Selected {len(selected_labels)} classes:")
print(selected_labels)

# --- Create new directories and copy images ---
def safe_copy(src_dir, dst_dir):
    """Copy files if src exists; return number of copied images."""
    if not os.path.exists(src_dir):
        return 0
    os.makedirs(dst_dir, exist_ok=True)
    count = 0
    for img in os.listdir(src_dir):
        s = os.path.join(src_dir, img)
        d = os.path.join(dst_dir, img)
        if os.path.isfile(s):
            shutil.copy2(s, d)
            count += 1
    return count

# Create and copy for both splits
for split in ["train", "validation"]:
    print(f"\n🔄 Copying {split} data...")
    for label in selected_labels:
        src = os.path.join(SOURCE_DIR, split, label)
        dst = os.path.join(DEST_DIR, split, label)
        copied = safe_copy(src, dst)
        print(f"{label:25s} → {copied:4d} images")

print("\n✅ Done! New subset created at:", DEST_DIR)

<h2>Import Libraries</h2>

In [ ]:
import torch.nn as nn
import torch
import torchvision
import torch.optim as optimizer
from torch.utils.data import random_split, DataLoader
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from torchvision import datasets
from torchvision.models.mobilenetv2 import MobileNetV2

In [ ]:
BATCH_SIZE = 32
TRAIN_DIR = "/kaggle/working/food-30/train"
TEST_DIR = "/kaggle/working/food-30/validation"
NUM_CLASSES = 30
IMG_SIZE = (260,260)

In [ ]:
from torchvision.models import EfficientNet_B2_Weights
weights = EfficientNet_B2_Weights.IMAGENET1K_V1

train_transforms = weights.transforms()  # baseline transforms


train_transforms = transforms.Compose([
    transforms.Resize(288),
    transforms.RandomResizedCrop(288, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_transforms.mean,
                         std=train_transforms.std)
])
valid_t = EfficientNet_B2_Weights.IMAGENET1K_V1.transforms()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
full_train_ds = ImageFolder(root=TRAIN_DIR,transform=train_transforms)
n_total = len(full_train_ds)
n_val = int(0.2*n_total)
n_train = n_total-n_val

train_ds ,valid_ds = random_split(full_train_ds,[n_train,n_val])
valid_ds.dataset.transform=valid_transforms

In [ ]:
test_ds = ImageFolder(root=TEST_DIR,transform=valid_transforms)

<h3>Create DataLoaders</h3>

In [ ]:
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
test_loader = DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False)
valid_loader = DataLoader(valid_ds,batch_size=BATCH_SIZE,shuffle=False)

In [ ]:
print(f"Length Train: {len(train_loader)}")
print(f"Length Test: {len(test_loader)}")
print(f"Length Validation: {len(valid_loader)}")

<h2>Pre-Train Model</h2>

In [ ]:
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
class FoodNet(nn.Module):
    def __init__(self,num_classes=NUM_CLASSES):
        super().__init__()
        base = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)
        self.backbone = base.features

        self.pool = nn.AdaptiveAvgPool2d(1)
        feat_dim = base.classifier[1].in_features
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim,128), #1280 is MobileNetV2 output channels
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,NUM_CLASSES)
        )

    def forward(self,x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x 
        

In [ ]:
model = FoodNet(num_classes=30).to(device)
for parameter in model.backbone.parameters():
    parameter.require_grad = False


In [ ]:
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
Epochs = 5
for i in range(Epochs):
    running_loss= 0.0
    correct= 0
    total=0
    for images,labels in train_loader:
        images , labels = images.to(device),labels.to(device)
        optimizer.zero_grad()
        labels_pred = model(images)
        loss = criterion(labels_pred,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*images.size(0)
        pred = labels_pred.argmax(1)
        correct +=(pred==labels).sum().item()
        total += labels.size(0)

    print(f"Epoch: {i+1} Training Accuracy: {100*correct/total:.4f}, Training Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
total=0
correct=0
val_loss=0.0
with torch.no_grad():
    for images,labels in valid_loader:
        images,labels = images.to(device),labels.to(device)
        y_pred = model(images)
        loss  = criterion(y_pred,labels)
        val_loss += loss.item()
        pred = y_pred.argmax(1)
        correct += (pred==labels).sum().item()
        total += labels.size(0)
    print(f"Validation Accuracy:{100*correct/total:.4f} Validation Loss: {val_loss/len(valid_loader):.4f}")